# Value-based vs Policy-based RL: **DQN vs PPO** on CartPole-v1

*A sibling notebook to the simple-RL companion. CPU-only, runs in a few minutes on free Colab.*

Reinforcement learning has two great families. This notebook puts one clean representative of
each, side by side, on the same toy task, with the **same environment-step budget**, so the
contrast is honest and visible.

| | **DQN** (this notebook) | **PPO** (this notebook) |
|---|---|---|
| Family | **Value-based** | **Policy-based** (actor-critic) |
| What it learns | action-value $Q(s,a)$; policy is the implicit $\arg\max_a Q$ | a parameterized stochastic policy $\pi_\theta(a\mid s)$ directly |
| Data usage | **off-policy** — a replay buffer recycles old transitions | **on-policy** — fresh rollouts, then the data is thrown away |
| Exploration | $\epsilon$-greedy (decay $\epsilon$) | entropy of the stochastic policy itself |
| Target / objective | TD target $y = r + \gamma\,\max_{a'} Q_{\text{target}}(s',a')$ | clipped surrogate + GAE($\lambda$) advantage + value baseline |
| Stability knobs | target network, replay buffer | trust-region clipping, multiple epochs per batch |
| Typical trade-off | **sample-efficient**, can be **less stable** | **stable**, but **sample-inefficient** (discards data) |

**The pedagogical point.** Both *solve* CartPole. But they sit at opposite ends of the design
space. DQN squeezes many gradient updates out of a *replay buffer* of past experience
(off-policy, sample-efficient, sometimes jumpy). PPO learns *on-policy* from freshly collected
rollouts and then discards them (stable, but it pays in samples). We plot return **versus
environment steps** — not versus iterations — so the sample-efficiency story is the thing you
actually see.

Notion deep-dives (theory companions): **[DQN & Rainbow](https://app.notion.com/p/37f95008d76681f396e7ffa88cb535e7)** &nbsp;·&nbsp; **[PPO](https://app.notion.com/p/37f95008d76681ae994febc1f7e89aa5)**

## 0. Setup

Install the (tiny) dependencies. On Colab this takes a few seconds.

In [ ]:
%pip install -q "gymnasium>=0.29" "pygame>=2.5"


### Imports and **global config**

Everything you might want to tweak lives in one place. The two methods are given the **same
environment-step budget** (`TOTAL_ENV_STEPS`) and the **same set of seeds**, which is what makes
the comparison fair. Small networks + CPU on purpose.

In [ ]:
import time, math, random, collections
from dataclasses import dataclass, field
from typing import List, Tuple

import numpy as np
import matplotlib
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Categorical

import gymnasium as gym

# ----- reproducibility helper ------------------------------------------------------------
def set_seed(seed: int):
    """Seed Python, NumPy and Torch so a run is repeatable."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

DEVICE = torch.device("cpu")          # free Colab: CPU is plenty for CartPole + tiny nets

# =====================================================================================
#                                 GLOBAL CONFIG
# =====================================================================================
ENV_ID          = "CartPole-v1"       # the shared task
GAMMA           = 0.99                # discount factor (shared)
SEEDS           = [0, 1, 2]           # average over a few seeds -> mean +/- std bands
SOLVED_RETURN   = 475.0              # CartPole-v1 is "solved" at avg return >= 475 (max 500)

# --- MATCHED BUDGET: both methods see exactly this many environment steps per seed --------
TOTAL_ENV_STEPS = 30_000             # small enough for a few-minute CPU run

# --- DQN (off-policy, value-based) hyperparameters ---------------------------------------
DQN_BUFFER_SIZE   = 20_000           # replay buffer capacity (off-policy reuse)
DQN_BATCH         = 64               # minibatch sampled from the buffer per update
DQN_LR            = 1e-3
DQN_TARGET_PERIOD = 500              # env steps between hard target-network syncs
DQN_LEARN_START   = 1_000           # warm up the buffer before learning
DQN_TRAIN_FREQ    = 1                # gradient step every this-many env steps
DQN_EPS_START     = 1.0             # epsilon-greedy: start fully random
DQN_EPS_END       = 0.02            # ...decay down to mostly-greedy
DQN_EPS_DECAY_STEPS = 8_000         # linear decay horizon (in env steps)
DOUBLE_DQN        = False            # <-- one-line toggle: Double-DQN target if True

# --- PPO (on-policy, policy-based) hyperparameters ---------------------------------------
PPO_ROLLOUT   = 2_000               # env steps collected per on-policy batch (then discarded)
PPO_EPOCHS    = 6                   # gradient passes over each fresh batch
PPO_MINIBATCH = 256
PPO_CLIP      = 0.2                 # clipped-surrogate epsilon (trust region)
PPO_LAM       = 0.95               # GAE(lambda)
PPO_LR        = 3e-3
PPO_ENT_COEF  = 0.0                 # entropy bonus (0 is fine for CartPole)
PPO_VF_COEF   = 0.5                 # value-loss weight

HIDDEN = 64                          # hidden width for ALL nets (kept tiny on purpose)

print("torch     :", torch.__version__)
print("gymnasium :", gym.__version__)
print("numpy     :", np.__version__)
print("device    :", DEVICE)
print("budget    :", TOTAL_ENV_STEPS, "env steps/method/seed   seeds =", SEEDS)


## 1. The task: CartPole-v1

A pole is hinged to a cart on a track. The agent pushes the cart **left (0)** or **right (1)**;
the goal is to keep the pole upright. State is 4-D: cart position, cart velocity, pole angle,
pole angular velocity. Reward is **+1 per timestep**, the episode ends when the pole falls or the
cart runs off — capped at **500** steps. "Solved" = average return $\ge 475$.

Below: a **random agent**. It topples almost immediately — that is our zero baseline.

In [ ]:
import io, base64
from IPython.display import HTML

def render_episode_frames(env_id, policy_fn, seed=0, max_steps=500):
    """Roll out one episode collecting rgb frames. `policy_fn(obs)->action`."""
    env = gym.make(env_id, render_mode="rgb_array")
    obs, _ = env.reset(seed=seed)
    frames = []
    for _ in range(max_steps):
        frames.append(env.render())
        a = policy_fn(obs)
        obs, r, term, trunc, _ = env.step(a)
        if term or trunc:
            break
    env.close()
    return frames

def frames_to_gif_html(frames, fps=30, scale=1.0):
    """Encode rgb frames -> an inline animated GIF (no extra deps beyond matplotlib/Pillow)."""
    try:
        from matplotlib import animation
        import matplotlib.pyplot as plt
        if not frames:
            return HTML("<i>(no frames)</i>")
        fig = plt.figure(figsize=(frames[0].shape[1] / 72 * scale,
                                   frames[0].shape[0] / 72 * scale))
        plt.axis("off")
        im = plt.imshow(frames[0])
        def _upd(i):
            im.set_data(frames[i]); return (im,)
        anim = animation.FuncAnimation(fig, _upd, frames=len(frames), interval=1000 / fps, blit=True)
        buf = io.BytesIO()
        anim.save(buf, writer="pillow", fps=fps)
        plt.close(fig)
        b64 = base64.b64encode(buf.getvalue()).decode("ascii")
        return HTML(f'<img src="data:image/gif;base64,{b64}"/>')
    except Exception as e:
        return HTML(f"<i>GIF rendering skipped: {e}</i>")

# random policy: ignore the observation, pick an action uniformly
def random_policy(env_id=ENV_ID):
    tmp = gym.make(env_id)
    n = tmp.action_space.n
    tmp.close()
    return lambda obs: int(np.random.randint(n))

_rand_frames = render_episode_frames(ENV_ID, random_policy(), seed=0)
print(f"random agent survived {len(_rand_frames)} steps")
frames_to_gif_html(_rand_frames, fps=30)


## 2. Networks

Both methods use a 2-layer MLP with width `HIDDEN`. The difference is the **head**:

- **DQN** maps `obs -> one Q-value per action`. The greedy policy is $\arg\max_a Q(s,a)$.
- **PPO** has a **policy head** (logits over actions, sampled via a `Categorical`) and a
  separate **value head** $V(s)$ used as the baseline. Here we use two small independent nets
  for clarity.

In [ ]:
class QNet(nn.Module):
    """DQN: state -> Q-value for each discrete action."""
    def __init__(self, obs_dim, n_actions, hidden=HIDDEN):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, n_actions),         # one output per action
        )
    def forward(self, x):
        return self.net(x)                        # shape: (batch, n_actions)


class PolicyNet(nn.Module):
    """PPO actor: state -> logits over actions -> Categorical distribution."""
    def __init__(self, obs_dim, n_actions, hidden=HIDDEN):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
            nn.Linear(hidden, n_actions),
        )
    def forward(self, x):
        return Categorical(logits=self.net(x))    # a torch distribution we can sample/score


class ValueNet(nn.Module):
    """PPO critic: state -> scalar state-value V(s) (the baseline)."""
    def __init__(self, obs_dim, hidden=HIDDEN):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
            nn.Linear(hidden, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)            # shape: (batch,)


## 3. Replay buffer (the off-policy ingredient)

DQN's defining feature: a ring buffer of past transitions $(s, a, r, s', \text{done})$. Each
gradient step samples a **random minibatch** from it, so a single experienced transition is
reused many times and across-time correlations are broken. PPO has no such thing — it learns
from one fresh batch and discards it.

In [ ]:
class ReplayBuffer:
    """Fixed-capacity ring buffer of transitions, with uniform random sampling."""
    def __init__(self, capacity, obs_dim):
        self.capacity = int(capacity)
        self.obs  = np.zeros((self.capacity, obs_dim), dtype=np.float32)
        self.next = np.zeros((self.capacity, obs_dim), dtype=np.float32)
        self.act  = np.zeros(self.capacity, dtype=np.int64)
        self.rew  = np.zeros(self.capacity, dtype=np.float32)
        self.done = np.zeros(self.capacity, dtype=np.float32)
        self.ptr  = 0          # next write index
        self.size = 0          # number of valid entries

    def push(self, s, a, r, s2, d):
        i = self.ptr
        self.obs[i]  = s
        self.act[i]  = a
        self.rew[i]  = r
        self.next[i] = s2
        self.done[i] = float(d)
        self.ptr  = (self.ptr + 1) % self.capacity   # wrap around (ring)
        self.size = min(self.size + 1, self.capacity)

    def sample(self, batch):
        idx = np.random.randint(0, self.size, size=batch)
        return (self.obs[idx], self.act[idx], self.rew[idx], self.next[idx], self.done[idx])

    def __len__(self):
        return self.size


## 4. `train_dqn(seed)`  — off-policy, value-based

The loop:
1. Act $\epsilon$-greedily (linearly decay $\epsilon$ over env steps), store the transition.
2. Once warmed up, every `DQN_TRAIN_FREQ` steps sample a minibatch and regress the online
   $Q$ toward the **TD target** $y = r + \gamma\,(1-\text{done})\max_{a'}Q_{\text{target}}(s',a')$.
3. Periodically **hard-copy** the online net into the target net (stabilizes the target).

*Double-DQN toggle:* if `DOUBLE_DQN`, the next action is selected by the **online** net and
evaluated by the **target** net — decoupling selection from evaluation reduces overestimation.

We log the **episodic return at the env-step index where each episode ended**, so the curve is
indexed by environment steps (comparable with PPO).

In [ ]:
def train_dqn(seed: int):
    set_seed(seed)
    env = gym.make(ENV_ID)
    obs, _ = env.reset(seed=seed)
    obs_dim   = env.observation_space.shape[0]
    n_actions = env.action_space.n

    online = QNet(obs_dim, n_actions).to(DEVICE)
    target = QNet(obs_dim, n_actions).to(DEVICE)
    target.load_state_dict(online.state_dict())     # start identical
    target.eval()
    opt = torch.optim.Adam(online.parameters(), lr=DQN_LR)

    buf = ReplayBuffer(DQN_BUFFER_SIZE, obs_dim)

    def epsilon(step):
        # linear decay from EPS_START to EPS_END over DQN_EPS_DECAY_STEPS env steps
        frac = min(1.0, step / DQN_EPS_DECAY_STEPS)
        return DQN_EPS_START + frac * (DQN_EPS_END - DQN_EPS_START)

    steps_axis, return_curve = [], []      # (env step, episodic return) at each episode end
    ep_return = 0.0

    for step in range(1, TOTAL_ENV_STEPS + 1):
        eps = epsilon(step)
        # ---- epsilon-greedy action selection ------------------------------------------
        if np.random.rand() < eps:
            a = int(np.random.randint(n_actions))
        else:
            with torch.no_grad():
                q = online(torch.as_tensor(obs, dtype=torch.float32, device=DEVICE).unsqueeze(0))
                a = int(q.argmax(dim=1).item())

        obs2, r, term, trunc, _ = env.step(a)
        done = term or trunc
        # store transition; `term` (not trunc) is the true terminal for bootstrapping
        buf.push(obs, a, r, obs2, term)
        obs = obs2
        ep_return += r

        if done:
            steps_axis.append(step)
            return_curve.append(ep_return)
            ep_return = 0.0
            obs, _ = env.reset()

        # ---- learning step ------------------------------------------------------------
        if len(buf) >= DQN_LEARN_START and step % DQN_TRAIN_FREQ == 0:
            s, ac, rw, s2, dn = buf.sample(DQN_BATCH)
            s   = torch.as_tensor(s,  device=DEVICE)
            s2  = torch.as_tensor(s2, device=DEVICE)
            ac  = torch.as_tensor(ac, device=DEVICE)
            rw  = torch.as_tensor(rw, device=DEVICE)
            dn  = torch.as_tensor(dn, device=DEVICE)

            # Q(s,a) for the actions actually taken
            q_sa = online(s).gather(1, ac.unsqueeze(1)).squeeze(1)

            with torch.no_grad():
                if DOUBLE_DQN:
                    next_a = online(s2).argmax(dim=1, keepdim=True)         # select w/ online
                    q_next = target(s2).gather(1, next_a).squeeze(1)        # evaluate w/ target
                else:
                    q_next = target(s2).max(dim=1).values                   # vanilla max
                y = rw + GAMMA * (1.0 - dn) * q_next                        # TD target

            loss = F.smooth_l1_loss(q_sa, y)        # Huber loss is the DQN default
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(online.parameters(), 10.0)
            opt.step()

        # ---- hard target sync ---------------------------------------------------------
        if step % DQN_TARGET_PERIOD == 0:
            target.load_state_dict(online.state_dict())

    env.close()
    # greedy policy closure for rendering
    def greedy_policy(o):
        with torch.no_grad():
            q = online(torch.as_tensor(o, dtype=torch.float32, device=DEVICE).unsqueeze(0))
            return int(q.argmax(dim=1).item())
    return np.array(steps_axis), np.array(return_curve), greedy_policy


## 5. `train_ppo(seed)`  — on-policy, policy-based

Repeat until the step budget is spent:
1. **Collect** a fresh rollout of `PPO_ROLLOUT` env steps with the *current* policy, recording
   `obs, action, log-prob, reward, value, done`.
2. **GAE($\lambda$)**: compute advantages
   $\hat A_t = \sum_l (\gamma\lambda)^l \delta_{t+l}$ with $\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$,
   and returns $\hat R_t = \hat A_t + V(s_t)$ as the value target.
3. **Optimize** for `PPO_EPOCHS` passes over the batch (minibatched), maximizing the **clipped
   surrogate** $\min(\rho_t \hat A_t,\ \text{clip}(\rho_t, 1\pm\epsilon)\hat A_t)$ where
   $\rho_t = \pi_\theta(a_t|s_t)/\pi_{\theta_\text{old}}(a_t|s_t)$, plus a value loss and optional
   entropy bonus. Then **discard the batch** and collect again — that's "on-policy".

In [ ]:
def compute_gae(rewards, values, dones, last_value, gamma=GAMMA, lam=PPO_LAM):
    """Generalized Advantage Estimation over one flat rollout (with episode boundaries via dones).
    `values` has length T; `last_value` is V(s_T) used to bootstrap the final step."""
    T = len(rewards)
    adv = np.zeros(T, dtype=np.float32)
    last_gae = 0.0
    for t in reversed(range(T)):
        next_v = last_value if t == T - 1 else values[t + 1]
        nonterminal = 1.0 - dones[t]            # if episode ended at t, don't bootstrap past it
        delta = rewards[t] + gamma * next_v * nonterminal - values[t]
        last_gae = delta + gamma * lam * nonterminal * last_gae
        adv[t] = last_gae
    returns = adv + np.asarray(values, dtype=np.float32)
    return adv, returns


def train_ppo(seed: int):
    set_seed(seed)
    env = gym.make(ENV_ID)
    obs, _ = env.reset(seed=seed)
    obs_dim   = env.observation_space.shape[0]
    n_actions = env.action_space.n

    policy = PolicyNet(obs_dim, n_actions).to(DEVICE)
    value  = ValueNet(obs_dim).to(DEVICE)
    opt = torch.optim.Adam(list(policy.parameters()) + list(value.parameters()), lr=PPO_LR)

    steps_axis, return_curve = [], []
    ep_return = 0.0
    global_step = 0

    while global_step < TOTAL_ENV_STEPS:
        # --------------------------------------------------- 1) collect a fresh rollout
        b_obs, b_act, b_logp, b_rew, b_val, b_done = [], [], [], [], [], []
        for _ in range(PPO_ROLLOUT):
            ot = torch.as_tensor(obs, dtype=torch.float32, device=DEVICE).unsqueeze(0)
            with torch.no_grad():
                dist = policy(ot)
                a = dist.sample()
                logp = dist.log_prob(a)
                v = value(ot)
            a_i = int(a.item())
            obs2, r, term, trunc, _ = env.step(a_i)
            done = term or trunc

            b_obs.append(obs); b_act.append(a_i); b_logp.append(float(logp.item()))
            b_rew.append(float(r)); b_val.append(float(v.item())); b_done.append(float(term))

            obs = obs2
            ep_return += r
            global_step += 1
            if done:
                steps_axis.append(global_step)
                return_curve.append(ep_return)
                ep_return = 0.0
                obs, _ = env.reset()
            if global_step >= TOTAL_ENV_STEPS:
                break

        # bootstrap value for the state we stopped at
        with torch.no_grad():
            last_v = float(value(torch.as_tensor(obs, dtype=torch.float32,
                                                 device=DEVICE).unsqueeze(0)).item())

        # --------------------------------------------------- 2) advantages via GAE
        adv, ret = compute_gae(np.array(b_rew), np.array(b_val), np.array(b_done), last_v)
        adv = (adv - adv.mean()) / (adv.std() + 1e-8)     # normalize advantages

        obs_t  = torch.as_tensor(np.array(b_obs), dtype=torch.float32, device=DEVICE)
        act_t  = torch.as_tensor(np.array(b_act), dtype=torch.int64,   device=DEVICE)
        logp_t = torch.as_tensor(np.array(b_logp), dtype=torch.float32, device=DEVICE)
        adv_t  = torch.as_tensor(adv, dtype=torch.float32, device=DEVICE)
        ret_t  = torch.as_tensor(ret, dtype=torch.float32, device=DEVICE)

        # --------------------------------------------------- 3) multiple epochs over batch
        N = obs_t.shape[0]
        idx = np.arange(N)
        for _ in range(PPO_EPOCHS):
            np.random.shuffle(idx)
            for start in range(0, N, PPO_MINIBATCH):
                mb = idx[start:start + PPO_MINIBATCH]
                dist = policy(obs_t[mb])
                new_logp = dist.log_prob(act_t[mb])
                entropy = dist.entropy().mean()

                ratio = torch.exp(new_logp - logp_t[mb])          # importance ratio rho_t
                unclipped = ratio * adv_t[mb]
                clipped = torch.clamp(ratio, 1 - PPO_CLIP, 1 + PPO_CLIP) * adv_t[mb]
                pi_loss = -torch.min(unclipped, clipped).mean()    # clipped surrogate (maximize)

                v_pred = value(obs_t[mb])
                v_loss = F.mse_loss(v_pred, ret_t[mb])             # value baseline regression

                loss = pi_loss + PPO_VF_COEF * v_loss - PPO_ENT_COEF * entropy
                opt.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(list(policy.parameters()) + list(value.parameters()), 0.5)
                opt.step()

    env.close()
    def greedy_policy(o):
        with torch.no_grad():
            dist = policy(torch.as_tensor(o, dtype=torch.float32, device=DEVICE).unsqueeze(0))
            return int(dist.probs.argmax(dim=1).item())   # deterministic = most-likely action
    return np.array(steps_axis), np.array(return_curve), greedy_policy


## 6. Run both methods across seeds

Same budget, same seeds. We keep the trained greedy policies for the demo GIF at the end. On
free Colab CPU this whole cell is a few minutes.

In [ ]:
def run_all():
    results = {"DQN": {"steps": [], "returns": []},
               "PPO": {"steps": [], "returns": []}}
    trained = {}
    t0 = time.time()
    for name, fn in [("DQN", train_dqn), ("PPO", train_ppo)]:
        for seed in SEEDS:
            ts = time.time()
            steps, rets, pol = fn(seed)
            results[name]["steps"].append(steps)
            results[name]["returns"].append(rets)
            trained[(name, seed)] = pol
            final = rets[-5:].mean() if len(rets) >= 5 else (rets.mean() if len(rets) else float("nan"))
            print(f"{name} seed={seed}: {len(rets):4d} episodes, "
                  f"final~{final:6.1f}, {time.time()-ts:5.1f}s")
    print(f"\nTOTAL elapsed: {time.time()-t0:.1f}s")
    return results, trained

results, trained = run_all()


## 7. The comparison plot: return **vs environment steps**

To put runs with different episode boundaries on a common axis, we interpolate each seed's
(step, return) curve onto a shared grid of environment steps, then plot the **mean ± std** band
across seeds. The dashed line is the solved threshold (475).

In [ ]:
def interp_to_grid(steps_list, returns_list, grid):
    """Interpolate each seed's curve onto a common env-step grid; return stacked array (seeds, grid)."""
    out = []
    for s, r in zip(steps_list, returns_list):
        if len(s) == 0:
            out.append(np.full_like(grid, np.nan, dtype=float)); continue
        # step-wise: value at grid point g is the most recent episodic return at step <= g
        out.append(np.interp(grid, s, r, left=r[0], right=r[-1]))
    return np.vstack(out)

def smooth(y, k=5):
    if len(y) < k: return y
    return np.convolve(y, np.ones(k) / k, mode="same")

grid = np.linspace(0, TOTAL_ENV_STEPS, 200)
plt.figure(figsize=(9, 5.5))
colors = {"DQN": "tab:red", "PPO": "tab:blue"}
finals = {}
for name in ["DQN", "PPO"]:
    M = interp_to_grid(results[name]["steps"], results[name]["returns"], grid)
    mean = np.nanmean(M, axis=0); std = np.nanstd(M, axis=0)
    mean_s = smooth(mean)
    plt.plot(grid, mean_s, color=colors[name], lw=2, label=f"{name} (mean over {len(SEEDS)} seeds)")
    plt.fill_between(grid, mean_s - std, mean_s + std, color=colors[name], alpha=0.18)
    finals[name] = np.nanmean(mean[-20:])

plt.axhline(SOLVED_RETURN, ls="--", color="gray", lw=1, label=f"solved = {SOLVED_RETURN:.0f}")
plt.xlabel("environment steps  (the fair, shared budget axis)")
plt.ylabel("episodic return")
plt.title("DQN (value-based, off-policy) vs PPO (policy-based, on-policy) on CartPole-v1")
plt.legend(loc="lower right"); plt.grid(alpha=0.3)

# auto annotation: which got higher faster, which is smoother (lower seed-variance)
def auc(name):
    M = interp_to_grid(results[name]["steps"], results[name]["returns"], grid)
    return np.nanmean(M)                       # area under mean curve ~ sample efficiency proxy
def jitter(name):
    M = interp_to_grid(results[name]["steps"], results[name]["returns"], grid)
    return np.nanmean(np.nanstd(M, axis=0))    # avg across-seed spread ~ (in)stability
more_eff   = "DQN" if auc("DQN")   > auc("PPO")   else "PPO"
more_smooth = "PPO" if jitter("PPO") < jitter("DQN") else "DQN"
note = (f"Sample efficiency (area under curve): {more_eff} reaches high return with fewer "
        f"env steps here.\nSmoothness (lower across-seed spread): {more_smooth} is steadier.")
plt.gcf().text(0.012, -0.06, note, fontsize=9, va="top")
plt.tight_layout(); plt.show()
print(note)
print({k: round(float(v), 1) for k, v in finals.items()})


## 8. A trained agent

Greedy rollout from one of the trained policies. It should balance the pole far longer than the
random agent at the top.

In [ ]:
# pick the best-performing (method, seed) for the demo
best_key, best_score = None, -1.0
for (name, seed), pol in trained.items():
    # score = final mean return of that run
    idx = SEEDS.index(seed)
    rr = results[name]["returns"][idx]
    sc = rr[-5:].mean() if len(rr) >= 5 else (rr.mean() if len(rr) else -1)
    if sc > best_score:
        best_score, best_key = sc, (name, seed)

print(f"demoing {best_key[0]} (seed={best_key[1]}), final~{best_score:.0f}")
demo_pol = trained[best_key]
_frames = render_episode_frames(ENV_ID, demo_pol, seed=123)
print(f"trained agent survived {len(_frames)} steps")
frames_to_gif_html(_frames, fps=30)

## 9. Takeaways & things to try

**When is value-based (DQN) preferable?**
- Discrete action spaces, where $\arg\max_a Q$ is cheap.
- When **sample efficiency** matters (real robots, expensive simulators): the replay buffer
  recycles every transition many times.
- Caveat: the "deadly triad" (function approximation + bootstrapping + off-policy) can make it
  **unstable** — hence target networks, Huber loss, and tricks like Double-DQN.

**When is policy-based (PPO) preferable?**
- **Continuous / large action spaces** (no max over actions needed; just parameterize $\pi$).
- When you want a **stochastic** policy or direct control of exploration via entropy.
- When **stability and reliability** beat raw sample count — PPO's clipping keeps each update in
  a trust region, which is why it's a default workhorse (and the backbone of RLHF/RL-for-LLMs).
- Cost: **on-policy** means fresh data each update; collected rollouts are used once and thrown
  away — sample-inefficient.

**Off-policy replay vs on-policy rollouts** is the crux. Replay = reuse old data (efficient,
riskier targets). On-policy = always-fresh data matched to the current policy (stable, wasteful).
Modern off-policy actor-critics (SAC, TD3) try to get PPO-like stability *with* replay-buffer
efficiency.

**Things to try**
- Flip `DOUBLE_DQN = True` — does the DQN curve get smoother / less optimistic?
- Shrink `DQN_BUFFER_SIZE` (e.g. 2,000) — too-small buffers reintroduce correlation and can hurt.
- Change `PPO_EPOCHS` (2 vs 10) — more epochs squeeze more from each batch but can over-fit it
  and break the trust-region assumption.
- Raise `TOTAL_ENV_STEPS` to 60k and watch both push toward 500.
- Try `PPO_ENT_COEF = 0.01` for stronger exploration; tune `PPO_CLIP`.

**Theory companions (Notion):**
[DQN & Rainbow](https://app.notion.com/p/37f95008d76681f396e7ffa88cb535e7) &nbsp;·&nbsp;
[PPO](https://app.notion.com/p/37f95008d76681ae994febc1f7e89aa5)